In [0]:
-- Insert values: chat table
MERGE INTO
	sac.support.chat s
USING (
	SELECT
		c.session_id,
		c.customer_id,
		c.agent_id,
		FIRST(m.classification) AS classification,
		FIRST(m.comment) AS comment
	FROM
		sac.support.chat_bronze c
			LEFT JOIN sac.support.ai_analyse_message m
				ON c.session_id = m.session_id
	GROUP BY
		c.session_id,
		c.customer_id,
		c.agent_id,
		c.ingestion_time
	QUALIFY
		row_number() OVER (PARTITION BY c.session_id ORDER BY c.ingestion_time DESC) = 1
) b
ON
	s.session_id = b.session_id
WHEN NOT MATCHED THEN INSERT *;

-- Insert values: message table
MERGE INTO
	sac.support.message s
USING (
	SELECT
		session_id,
		speaker,
		timestamp,
		message,
		classification,
		comment,
		CASE
			WHEN sentiment IN ('Positive', 'Positiv', 'Positives') THEN 'Positiv'
			WHEN sentiment IN ('Neutral', 'Neutrales') THEN 'Neutral'
			WHEN sentiment IN ('Negative', 'Negativ', 'Negatives') THEN 'Negativ'
			ELSE 'Unbekannt'
		END AS sentiment
	FROM
		sac.support.ai_analyse_message
	QUALIFY
		row_number() OVER (PARTITION BY session_id, timestamp ORDER BY ingestion_time DESC) = 1
) b
ON
	b.session_id = s.session_id
	AND b.speaker = s.speaker
	AND b.timestamp = s.timestamp
WHEN NOT MATCHED THEN INSERT *;